# `tm_scraper` — Transfermarkt Football Data Lake

## Strategy

### Goal
Scrape all available match data from Transfermarkt across a defined set of leagues, from the earliest available season to the present. The output is a local SQLite database (`tm_football.db`) that serves as a personal football data lake — reusable across projects, queryable without loading everything into memory.

The immediate use case is the *a corto muso* article (manager result distributions), but the schema is intentionally richer than that project alone requires.

### Leagues in scope

| Code | League | Country |
|------|--------|---------|
| GB1  | Premier League | England |
| IT1  | Serie A | Italy |
| ES1  | La Liga | Spain |
| L1   | Bundesliga | Germany |
| FR1  | Ligue 1 | France |
| PO1  | Primeira Liga | Portugal |
| NL1  | Eredivisie | Netherlands |
| TR1  | Süper Lig | Turkey |

### Two-stage scraping architecture

**Stage 1 — Game index** (fast: ~1 request per league per season)

```
URL: transfermarkt.com/{slug}/gesamtspielplan/wettbewerb/{CODE}/saison_id/{YEAR}
Extracts: game_id, date, home_club, away_club, score, round
Output: game_index table in tm_football.db
Runtime: ~5 minutes for all leagues and seasons
```

Stage 1 is run once in full before Stage 2 begins. This gives us the complete queue of game IDs upfront — Stage 2 is then a pure resumable job with no structural dependencies.

**Stage 2 — Match reports** (slow: ~1 request per match)

```
URL: transfermarkt.com/spielbericht/index/spielbericht/{game_id}
Extracts: home_manager, away_manager, formations, attendance, referee
Output: matches table in tm_football.db
Runtime: several hours (run overnight, fully resumable)
```

Every match report write uses `INSERT OR IGNORE` — re-running Stage 2 is always safe and will only fetch IDs not yet in the database.

### Execution plan

| Block | Content |
|-------|---------|
| A | Setup: imports, DB schema, helpers |
| B | Stage 1: build complete game index |
| C | POC: scrape one full season end-to-end (feasibility validation) |
| D | Stage 2: last 15 seasons, all leagues (first usable dataset) |
| E | Stage 2: full archive, all available seasons |

### Design principles

- **Incremental writes**: every row is committed immediately; a crash loses at most one match
- **Resumable**: Stage 2 skips any `game_id` already present in the database
- **Polite**: 2-second sleep between requests; randomised ±0.5s jitter to avoid pattern detection
- **Single artefact**: one `.db` file, portable, queryable with pandas via `pd.read_sql`
- **Rich schema**: formation, attendance, referee captured at no extra cost — free future value

## Block A — Setup

Imports, configuration, database initialisation, and shared HTTP/parsing helpers.
Run this cell first in every session before any other block.

In [1]:
# ── A.1  Imports ─────────────────────────────────────────────────────────────
import sqlite3
import time
import random
import logging
from datetime import datetime
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

print("Imports OK")

Imports OK


In [49]:
# ── A.2  Configuration ────────────────────────────────────────────────────────

DB_PATH = Path("tm_football.db")   # change path if you want it elsewhere

LEAGUES = {
    "GB1": "premier-league",
    "IT1": "serie-a",
    "ES1": "laliga",
    "L1" : "bundesliga",
    "FR1": "ligue-1",
    "PO1": "liga-nos",
    "NL1": "eredivisie",
    "TR1": "super-lig",
}

SEASON_START = 1993   # earliest Transfermarkt has reliable data
SEASON_END   = 2025   # inclusive; update each year

BASE_URL  = "https://www.transfermarkt.com"
HEADERS   = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}
# SLEEP_BASE   = 2.0   # seconds between requests
# SLEEP_JITTER = 0.5   # ± random jitter added to SLEEP_BASE

SLEEP_BASE   = 1.0
SLEEP_JITTER = 0.2


print(f"DB path     : {DB_PATH.resolve()}")
print(f"Leagues     : {list(LEAGUES.keys())}")
print(f"Season range: {SEASON_START} – {SEASON_END}")

DB path     : C:\Users\pacor\Documents\Notebooks\Python\Inpat Expat\inpat-expat\corto-muso\tm_football.db
Leagues     : ['GB1', 'IT1', 'ES1', 'L1', 'FR1', 'PO1', 'NL1', 'TR1']
Season range: 1993 – 2025


In [3]:
# ── A.3  Database initialisation ──────────────────────────────────────────────
# Running this cell is idempotent: safe to re-run, never drops existing data.

def init_db(path: Path) -> sqlite3.Connection:
    con = sqlite3.connect(path)
    con.execute("PRAGMA journal_mode=WAL")   # safer for incremental writes
    con.execute("PRAGMA foreign_keys=ON")
    con.executescript("""
        CREATE TABLE IF NOT EXISTS game_index (
            game_id      TEXT PRIMARY KEY,
            league_code  TEXT NOT NULL,
            season       INTEGER NOT NULL,
            round        TEXT,
            date         TEXT,
            home_club    TEXT,
            away_club    TEXT,
            home_goals   INTEGER,
            away_goals   INTEGER,
            indexed_at   TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS matches (
            game_id          TEXT PRIMARY KEY,
            league_code      TEXT NOT NULL,
            season           INTEGER NOT NULL,
            round            TEXT,
            date             TEXT,
            home_club        TEXT,
            away_club        TEXT,
            home_goals       INTEGER,
            away_goals       INTEGER,
            home_manager     TEXT,
            away_manager     TEXT,
            home_formation   TEXT,
            away_formation   TEXT,
            attendance       INTEGER,
            referee          TEXT,
            scraped_at       TEXT NOT NULL
        );

        CREATE INDEX IF NOT EXISTS idx_matches_league_season
            ON matches (league_code, season);
        CREATE INDEX IF NOT EXISTS idx_matches_home_manager
            ON matches (home_manager);
        CREATE INDEX IF NOT EXISTS idx_matches_away_manager
            ON matches (away_manager);
    """)
    con.commit()
    return con


con = init_db(DB_PATH)
print(f"Database initialised: {DB_PATH.resolve()}")

# Quick sanity check
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)
print(f"Tables: {tables['name'].tolist()}")

Database initialised: C:\Users\pacor\Documents\Notebooks\Python\Inpat Expat\inpat-expat\corto-muso\tm_football.db
Tables: ['game_index', 'matches']


In [4]:
# ── A.4  Shared HTTP helper ───────────────────────────────────────────────────

session = requests.Session()
session.headers.update(HEADERS)

def fetch(url: str, retries: int = 3) -> BeautifulSoup | None:
    """Fetch a Transfermarkt page and return a BeautifulSoup object.
    Returns None on persistent failure. Sleeps politely after every request.
    """
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15)
            resp.raise_for_status()
            sleep = SLEEP_BASE + random.uniform(-SLEEP_JITTER, SLEEP_JITTER)
            time.sleep(sleep)
            return BeautifulSoup(resp.content, "html.parser")
        except requests.RequestException as e:
            log.warning(f"Attempt {attempt}/{retries} failed for {url}: {e}")
            if attempt < retries:
                time.sleep(SLEEP_BASE * attempt * 2)   # back off on retry
    log.error(f"All {retries} attempts failed for {url}")
    return None


print("HTTP session ready")

HTTP session ready


## Block B — Stage 1: Build game index

Fetches one schedule page per (league, season) and extracts all game IDs with basic metadata. Fast — typically under 10 minutes for all leagues and seasons.

Safe to re-run: uses `INSERT OR IGNORE`, so already-indexed games are skipped.

In [9]:
# ── B.1  Schedule page parser ────────────────────────────────────────────────

def parse_schedule(soup: BeautifulSoup, league_code: str, season: int) -> list[dict]:
    """Extract all matches from a Transfermarkt season schedule page."""
    rows = []
    # Each match row links to the spielbericht page; the game_id is in that URL
    #for a_tag in soup.select("a[href*='/spielbericht/index/spielbericht/']"):
    for a_tag in soup.select("a[href*='/index/spielbericht/']"):
        href = a_tag.get("href", "")
        try:
            game_id = href.split("/spielbericht/")[-1].split("/")[0]
            if not game_id.isdigit():
                continue
        except (IndexError, AttributeError):
            continue

        # Walk up to the table row to get date, teams, score
        tr = a_tag.find_parent("tr")
        if tr is None:
            continue
        cells = tr.find_all("td")
        if len(cells) < 4:
            continue

        # Extract text safely
        def cell_text(i):
            return cells[i].get_text(strip=True) if i < len(cells) else ""

        date       = cell_text(1)
        round_     = cell_text(0)
        home_club  = cell_text(2)
        score_raw  = cell_text(3)
        away_club  = cell_text(4)

        # Parse score e.g. "2:1" or "2:1 n.V."
        home_goals = away_goals = None
        score_part = score_raw.split(" ")[0]  # drop extra time / pens notes
        if ":" in score_part:
            parts = score_part.split(":")
            try:
                home_goals = int(parts[0])
                away_goals = int(parts[1])
            except ValueError:
                pass

        rows.append({
            "game_id"    : game_id,
            "league_code": league_code,
            "season"     : season,
            "round"      : round_,
            "date"       : date,
            "home_club"  : home_club,
            "away_club"  : away_club,
            "home_goals" : home_goals,
            "away_goals" : away_goals,
            "indexed_at" : datetime.utcnow().isoformat(),
        })

    # Deduplicate by game_id (same link may appear multiple times on page)
    seen = set()
    unique = []
    for r in rows:
        if r["game_id"] not in seen:
            seen.add(r["game_id"])
            unique.append(r)
    return unique


print("Schedule parser defined")

Schedule parser defined


In [10]:
# ── B.2  Index one (league, season) — test on a single season first ──────────

def index_season(league_code: str, season: int, con: sqlite3.Connection) -> int:
    """Fetch and store the game index for one league/season.
    Returns number of new rows inserted.
    """
    slug = LEAGUES[league_code]
    url  = f"{BASE_URL}/{slug}/gesamtspielplan/wettbewerb/{league_code}/saison_id/{season}"
    soup = fetch(url)
    if soup is None:
        log.warning(f"Skipping {league_code} {season} — fetch failed")
        return 0

    games = parse_schedule(soup, league_code, season)
    if not games:
        log.warning(f"No games found for {league_code} {season} — season may not exist")
        return 0

    cur = con.cursor()
    cur.executemany(
        """
        INSERT OR IGNORE INTO game_index
            (game_id, league_code, season, round, date,
             home_club, away_club, home_goals, away_goals, indexed_at)
        VALUES
            (:game_id, :league_code, :season, :round, :date,
             :home_club, :away_club, :home_goals, :away_goals, :indexed_at)
        """,
        games,
    )
    con.commit()
    inserted = cur.rowcount
    log.info(f"{league_code} {season}: {len(games)} games found, {inserted} new")
    return inserted


# ── Smoke test: index one season before running the full loop ─────────────────
print("Smoke test: Serie A 2023/24")
n = index_season("IT1", 2023, con)
print(f"Inserted: {n}")

# Check what landed in the DB
pd.read_sql(
    "SELECT * FROM game_index WHERE league_code='IT1' AND season=2023 LIMIT 5",
    con
)

Smoke test: Serie A 2023/24


11:27:14  INFO  IT1 2023: 380 games found, 380 new


Inserted: 380


,game_id,league_code,season,round,date,home_club,away_club,home_goals,away_goals,indexed_at
0,4103473,IT1,2023,Sat19/08/23,6:30 PM,FC Empoli,0:1,None,None,2026-06-01T10:27:14.833929
1,4103474,IT1,2023,,,Frosinone,1:3,None,None,2026-06-01T10:27:14.833929
2,4103475,IT1,2023,,8:45 PM,Genoa,1:4,None,None,2026-06-01T10:27:14.833929
3,4103476,IT1,2023,,,Inter,2:0,None,None,2026-06-01T10:27:14.833929
4,4103478,IT1,2023,Sun20/08/23,6:30 PM,Roma,2:2,None,None,2026-06-01T10:27:14.833929


In [12]:
# # ── B.2 DIAGNOSTIC: inspect the raw HTML of the schedule page ────────────────
# slug = LEAGUES["IT1"]
# url  = f"{BASE_URL}/{slug}/gesamtspielplan/wettbewerb/IT1/saison_id/2023"
# soup = fetch(url)

# # 1. Check the page title to confirm we got the right page
# print("Page title:", soup.title.string if soup.title else "NO TITLE")
# print()

# # 2. Count ALL anchor tags with 'spielbericht' in href
# spielbericht_links = soup.find_all("a", href=lambda h: h and "spielbericht" in h)
# print(f"Anchor tags containing 'spielbericht': {len(spielbericht_links)}")
# if spielbericht_links:
#     print("First 3 hrefs:")
#     for a in spielbericht_links[:3]:
#         print(f"  {a.get('href')}")
# print()

# # 3. If none found, show ALL unique href patterns to find the right one
# if not spielbericht_links:
#     all_hrefs = [a.get("href","") for a in soup.find_all("a", href=True)]
#     patterns = set(h.split("/")[1] if "/" in h else h for h in all_hrefs if h)
#     print("Unique href first-segment patterns found on page:")
#     for p in sorted(patterns)[:30]:
#         print(f"  /{p}/...")

In [11]:
# ── B.3  Full Stage 1: all leagues, all seasons ───────────────────────────────
# Only run this after the smoke test in B.2 looks correct.
# Runtime: ~10 minutes. Safe to interrupt and re-run.

total_new = 0
for league_code in LEAGUES:
    for season in range(SEASON_START, SEASON_END + 1):
        # Skip if already fully indexed for this league/season
        existing = pd.read_sql(
            "SELECT COUNT(*) AS n FROM game_index WHERE league_code=? AND season=?",
            con, params=(league_code, season)
        ).iloc[0]["n"]
        if existing > 0:
            log.info(f"{league_code} {season}: already indexed ({existing} games), skipping")
            continue
        total_new += index_season(league_code, season, con)

print(f"\nStage 1 complete. New rows inserted this run: {total_new}")
pd.read_sql(
    "SELECT league_code, season, COUNT(*) as games FROM game_index GROUP BY league_code, season ORDER BY league_code, season",
    con
)

11:27:24  INFO  GB1 1993: 462 games found, 462 new
11:27:30  INFO  GB1 1994: 462 games found, 462 new
11:27:34  INFO  GB1 1995: 380 games found, 380 new
11:27:38  INFO  GB1 1996: 380 games found, 380 new
11:27:42  INFO  GB1 1997: 380 games found, 380 new
11:27:47  INFO  GB1 1998: 380 games found, 380 new
11:27:51  INFO  GB1 1999: 380 games found, 380 new
11:27:55  INFO  GB1 2000: 380 games found, 380 new
11:28:00  INFO  GB1 2001: 380 games found, 380 new
11:28:03  INFO  GB1 2002: 380 games found, 380 new
11:28:07  INFO  GB1 2003: 380 games found, 380 new
11:28:11  INFO  GB1 2004: 380 games found, 380 new
11:28:17  INFO  GB1 2005: 380 games found, 380 new
11:28:25  INFO  GB1 2006: 380 games found, 380 new
11:28:30  INFO  GB1 2007: 380 games found, 380 new
11:28:36  INFO  GB1 2008: 380 games found, 380 new
11:28:40  INFO  GB1 2009: 380 games found, 380 new
11:28:45  INFO  GB1 2010: 380 games found, 380 new
11:28:50  INFO  GB1 2011: 380 games found, 380 new
11:28:55  INFO  GB1 2012: 380 g


Stage 1 complete. New rows inserted this run: 88659


,league_code,season,games
0,ES1,1993,380
1,ES1,1994,380
2,ES1,1995,462
3,ES1,1996,462
4,ES1,1997,380
...,...,...,...
259,TR1,2021,380
260,TR1,2022,342
261,TR1,2023,380
262,TR1,2024,342


## Block C — POC: Stage 2 on one full season

Before running the full Stage 2 overnight, validate that the match report parser
works correctly and that we are not getting rate-limited.

Target: Serie A 2023/24 (~380 matches). Expected runtime: ~15 minutes.

In [36]:
def parse_match_report(soup: BeautifulSoup, game_id: str, league_code: str, season: int) -> dict | None:

    # ── Managers ──────────────────────────────────────────────────────────────
    home_manager = away_manager = None
    manager_rows = []
    for td in soup.find_all("div", string=lambda t: t and "Manager:" in t):
        tr = td.parent.parent
        name_td = tr.find("td", class_="bench-table__td")
        if name_td:
            a = name_td.find("a")
            if a:
                manager_rows.append(a.get_text(strip=True))
    if len(manager_rows) > 0:
        home_manager = manager_rows[0]
    if len(manager_rows) > 1:
        away_manager = manager_rows[1]

    # ── Formations ────────────────────────────────────────────────────────────
    home_formation = away_formation = None
    formation_divs = soup.select("div.formation-subtitle")
    def extract_formation(text):
        return text.split(":")[-1].strip() if ":" in text else text.strip()
    if len(formation_divs) > 0:
        home_formation = extract_formation(formation_divs[0].get_text(strip=True))
    if len(formation_divs) > 1:
        away_formation = extract_formation(formation_divs[1].get_text(strip=True))

    # ── Attendance ────────────────────────────────────────────────────────────
    attendance = None
    for strong in soup.find_all("strong"):
        text = strong.get_text(strip=True)
        if "Attendance" in text or "Zuschauer" in text:
            raw = text.split(":")[-1].strip()
            try:
                attendance = int(raw.replace(".", "").replace(",", ""))
            except ValueError:
                pass
            break

    # ── Referee ───────────────────────────────────────────────────────────────
    referee = None
    for strong in soup.find_all("strong"):
        if "Referee" in strong.get_text() or "Schiedsrichter" in strong.get_text():
            for sib in strong.next_siblings:
                if hasattr(sib, "select_one"):   # only real Tag objects, not strings
                    a = sib.select_one("a")
                    if a:
                        referee = a.get_text(strip=True)
                        break
            break    # ── Score ─────────────────────────────────────────────────────────────────
    home_goals = away_goals = None
    score_tag = soup.select_one("div.sb-endstand")
    if score_tag:
        score_text = score_tag.get_text(strip=True).split("(")[0].strip()
        if ":" in score_text:
            parts = score_text.split(":")
            try:
                home_goals = int(parts[0])
                away_goals = int(parts[1])
            except ValueError:
                pass

    # ── Clubs ─────────────────────────────────────────────────────────────────
    home_club = away_club = None
    verein_links = soup.find_all(
        "a", href=lambda h: h and "/startseite/verein/" in h
    )
    name_links = [a for a in verein_links if a.get_text(strip=True)]
    if len(name_links) > 0:
        home_club = name_links[0].get_text(strip=True)
    if len(name_links) > 1:
        away_club = name_links[1].get_text(strip=True)

    # ── Date ──────────────────────────────────────────────────────────────────
    date = None
    date_tag = soup.select_one("p.sb-datum")
    if date_tag:
        date = date_tag.get_text(strip=True)

    return {
        "game_id"        : game_id,
        "league_code"    : league_code,
        "season"         : season,
        "round"          : None,
        "date"           : date,
        "home_club"      : home_club,
        "away_club"      : away_club,
        "home_goals"     : home_goals,
        "away_goals"     : away_goals,
        "home_manager"   : home_manager,
        "away_manager"   : away_manager,
        "home_formation" : home_formation,
        "away_formation" : away_formation,
        "attendance"     : attendance,
        "referee"        : referee,
        "scraped_at"     : datetime.utcnow().isoformat(),
    }

In [37]:
# ── C.2  DB insert helper ─────────────────────────────────────────────────────

INSERT_SQL = """
INSERT OR IGNORE INTO matches
    (game_id, league_code, season, round, date,
     home_club, away_club, home_goals, away_goals,
     home_manager, away_manager,
     home_formation, away_formation,
     attendance, referee, scraped_at)
VALUES
    (:game_id, :league_code, :season, :round, :date,
     :home_club, :away_club, :home_goals, :away_goals,
     :home_manager, :away_manager,
     :home_formation, :away_formation,
     :attendance, :referee, :scraped_at)
"""

def scrape_match(game_id: str, league_code: str, season: int, con: sqlite3.Connection) -> bool:
    """Scrape one match report and write to DB. Returns True on success."""
    url  = f"{BASE_URL}/spielbericht/index/spielbericht/{game_id}"
    soup = fetch(url)
    if soup is None:
        return False
    row = parse_match_report(soup, game_id, league_code, season)
    if row is None:
        return False
    con.execute(INSERT_SQL, row)
    con.commit()
    return True


print("Insert helper defined")

Insert helper defined


In [38]:
# ── C.3  Single-match smoke test ──────────────────────────────────────────────
# Fetch ONE match report to verify the HTML selectors work before
# running anything at scale. Inspect the output carefully.

sample = pd.read_sql(
    "SELECT game_id, league_code, season FROM game_index WHERE league_code='IT1' AND season=2023 LIMIT 1",
    con
).iloc[0]

print(f"Testing game_id: {sample.game_id}")
url  = f"{BASE_URL}/spielbericht/index/spielbericht/{sample.game_id}"
soup = fetch(url)

if soup:
    row = parse_match_report(soup, sample.game_id, sample.league_code, int(sample.season))
    print("\nParsed row:")
    for k, v in row.items():
        print(f"  {k:20s}: {v}")
else:
    print("Fetch failed — check connection or User-Agent")

Testing game_id: 4103473

Parsed row:
  game_id             : 4103473
  league_code         : IT1
  season              : 2023
  round               : None
  date                : 1. Matchday|Sat, 19/08/23|  6:30 PM
  home_club           : FC Empoli
  away_club           : Hellas Verona
  home_goals          : 0
  away_goals          : 1
  home_manager        : Paolo Zanetti
  away_manager        : Marco Baroni
  home_formation      : 4-2-3-1
  away_formation      : 3-4-1-2
  attendance          : 7940
  referee             : Luca Massimi
  scraped_at          : 2026-06-01T11:02:50.068249


In [39]:
# for strong in soup.find_all("strong"):
#     if "Referee" in strong.get_text():
#         print(f"strong text: '{strong.get_text()}'")
#         print(f"next_sibling type: {type(strong.next_sibling)}")
#         print(f"next_sibling repr: {repr(strong.next_sibling)}")
#         # Try next siblings further along
#         for i, sib in enumerate(strong.next_siblings):
#             print(f"  sibling {i}: type={type(sib).__name__}  repr={repr(str(sib)[:80])}")
#             if i > 4:
#                 break
#         break

In [40]:
# # ── C.3 DIAGNOSTIC: inspect actual HTML structure of a match report ───────────
# url  = f"{BASE_URL}/spielbericht/index/spielbericht/4103473"
# soup = fetch(url)

# # 1. Manager — search broadly for anything containing a known manager name
# print("=== MANAGER SEARCH ===")
# # Davide Nicola managed Empoli that season — search for his name
# for tag in soup.find_all(string=lambda t: t and "Nicola" in t):
#     parent = tag.parent
#     print(f"  Tag: <{parent.name} class='{parent.get('class', '')}'>  text: {tag.strip()[:80]}")

# print()

# # 2. Formation — search for typical formation strings like "4-3-3" or "3-5-2"
# print("=== FORMATION SEARCH ===")
# import re
# for tag in soup.find_all(string=re.compile(r'\d-\d-\d')):
#     parent = tag.parent
#     print(f"  Tag: <{parent.name} class='{parent.get('class', '')}'>  text: {tag.strip()[:80]}")

# print()

# # 3. Attendance — search for the attendance figure (7940 from the referee field)
# print("=== ATTENDANCE SEARCH ===")
# for tag in soup.find_all(string=lambda t: t and "7.940" in t or (t and "7,940" in t)):
#     parent = tag.parent
#     print(f"  Tag: <{parent.name} class='{parent.get('class', '')}'>  text: {tag.strip()[:80]}")

# print()

# # 4. Home club — what's in the sb-team divs?
# print("=== CLUB TAGS (sb-team) ===")
# for tag in soup.select("div.sb-team"):
#     print(f"  {tag.get_text(strip=True)[:80]}")

# print()

# # 5. Dump all unique class names containing 'trainer' or 'aufstellung' or 'formation'
# print("=== RELEVANT CSS CLASSES ===")
# classes = set()
# for tag in soup.find_all(True):
#     for c in tag.get("class", []):
#         if any(x in c.lower() for x in ["trainer", "aufstellung", "formation", "coach", "lineup"]):
#             classes.add(c)
# print(classes)

In [41]:
# # Find the aufstellung-box and see what's inside it
# print("=== AUFSTELLUNG-BOX STRUCTURE ===")
# boxes = soup.select("div.aufstellung-box")
# for i, box in enumerate(boxes[:2]):  # home and away box
#     print(f"\n--- Box {i} ---")
#     # Find all <a> tags inside each box
#     links = box.find_all("a", href=True)
#     for a in links[:5]:
#         print(f"  href: {a.get('href','')[:60]}  text: {a.get_text(strip=True)[:40]}")
#     # Find the trainer/coach section specifically
#     trainer = box.select("div.aufstellung-unterueberschrift-mannschaft")
#     for t in trainer:
#         print(f"  unterueberschrift: {t.get_text(strip=True)[:80]}")

In [42]:
# # Search for the trainer/coach section outside the formation box
# print("=== TRAINER/COACH SECTION ===")

# # Try the unterueberschrift divs more broadly
# for tag in soup.find_all("div", class_="aufstellung-unterueberschrift-mannschaft"):
#     print(f"  text: {tag.get_text(strip=True)[:80]}")
#     # Check siblings and parent
#     parent = tag.parent
#     print(f"  parent class: {parent.get('class', '')}")
#     all_text = parent.get_text(" ", strip=True)
#     print(f"  parent text: {all_text[:150]}")
#     print()

# print("=== SEARCH FOR COACH LABEL ===")
# # Search for any tag containing "Coach" or "Trainer" or "Manager" label
# for tag in soup.find_all(string=lambda t: t and any(
#     x in t for x in ["Coach:", "Trainer:", "Manager:", "Head coach"]
# )):
#     parent = tag.parent
#     print(f"  <{parent.name} class='{parent.get('class','')}'>: {tag.strip()[:100]}")
#     # Also print the next sibling
#     nxt = parent.find_next_sibling()
#     if nxt:
#         print(f"    next sibling: {nxt.get_text(strip=True)[:80]}")

In [43]:
# print("=== MANAGER DIV + SIBLINGS ===")
# for tag in soup.find_all("div", string=lambda t: t and "Manager:" in t):
#     print(f"  label div text: '{tag.get_text(strip=True)}'")
#     nxt = tag.find_next_sibling()
#     if nxt:
#         print(f"  next sibling: <{nxt.name} class='{nxt.get('class','')}'>: '{nxt.get_text(strip=True)[:80]}'")
#     # Also check parent for context
#     parent = tag.parent
#     print(f"  parent class: {parent.get('class', '')}")
#     print(f"  parent text: {parent.get_text(' ', strip=True)[:150]}")
#     print()

In [44]:
# print("=== MANAGER FULL CONTEXT ===")
# for tag in soup.find_all("div", string=lambda t: t and "Manager:" in t):
#     # Walk up two levels and dump the full HTML chunk
#     grandparent = tag.parent.parent
#     print(f"  grandparent class: {grandparent.get('class', '')}")
#     print(f"  grandparent HTML:\n{grandparent}")
#     print()

In [45]:
# ── C.4  POC: full season scrape (Serie A 2023/24) ────────────────────────────
# Only run after C.3 shows correct manager names.
# Expected runtime: ~15 minutes. Watch for None manager values after ~50 matches
# as an early signal of rate limiting.

def scrape_season(league_code: str, season: int, con: sqlite3.Connection) -> dict:
    """Scrape all unscraped matches for one league/season.
    Returns a summary dict with counts.
    """
    # Only fetch game IDs not yet in the matches table
    pending = pd.read_sql(
        """
        SELECT gi.game_id, gi.league_code, gi.season
        FROM game_index gi
        LEFT JOIN matches m USING (game_id)
        WHERE gi.league_code = ? AND gi.season = ? AND m.game_id IS NULL
        """,
        con, params=(league_code, season)
    )

    total   = len(pending)
    success = 0
    failed  = 0

    log.info(f"{league_code} {season}: {total} matches to scrape")

    for i, row in enumerate(pending.itertuples(), 1):
        ok = scrape_match(row.game_id, row.league_code, int(row.season), con)
        if ok:
            success += 1
        else:
            failed += 1
        if i % 50 == 0:
            log.info(f"  {i}/{total} done  ({success} ok, {failed} failed)")

    log.info(f"{league_code} {season}: finished — {success} ok, {failed} failed")
    return {"league": league_code, "season": season,
            "total": total, "success": success, "failed": failed}


# Run the POC
result = scrape_season("IT1", 2023, con)
print(result)

# Inspect a sample of what landed in the DB
pd.read_sql(
    """
    SELECT game_id, date, home_club, away_club,
           home_goals, away_goals, home_manager, away_manager, home_formation
    FROM matches
    WHERE league_code='IT1' AND season=2023
    LIMIT 10
    """,
    con
)

12:03:58  INFO  IT1 2023: 380 matches to scrape
12:07:13  INFO    50/380 done  (50 ok, 0 failed)
12:10:02  INFO    100/380 done  (100 ok, 0 failed)
12:12:28  INFO    150/380 done  (150 ok, 0 failed)
12:15:06  INFO    200/380 done  (200 ok, 0 failed)
12:17:39  INFO    250/380 done  (250 ok, 0 failed)
12:20:09  INFO    300/380 done  (300 ok, 0 failed)
12:23:10  INFO    350/380 done  (350 ok, 0 failed)
12:24:44  INFO  IT1 2023: finished — 380 ok, 0 failed


{'league': 'IT1', 'season': 2023, 'total': 380, 'success': 380, 'failed': 0}


,game_id,date,home_club,away_club,home_goals,away_goals,home_manager,away_manager,home_formation
0,4103473,"1. Matchday|Sat, 19/08/23| 6:30 PM",FC Empoli,Hellas Verona,0,1,Paolo Zanetti,Marco Baroni,4-2-3-1
1,4103474,"1. Matchday|Sat, 19/08/23| 6:30 PM",Frosinone Calcio,SSC Napoli,1,3,Eusebio Di Francesco,Rudi Garcia,4-3-3 Attacking
2,4103475,"1. Matchday|Sat, 19/08/23| 8:45 PM",Genoa CFC,ACF Fiorentina,1,4,Alberto Gilardino,Vincenzo Italiano,3-5-2 flat
3,4103476,"1. Matchday|Sat, 19/08/23| 8:45 PM",Inter Milan,AC Monza,2,0,Simone Inzaghi,Raffaele Palladino,3-5-2 flat
4,4103478,"1. Matchday|Sun, 20/08/23| 6:30 PM",AS Roma,US Salernitana 1919,2,2,José Mourinho,Paulo Sousa,3-5-2 flat
5,4103479,"1. Matchday|Sun, 20/08/23| 6:30 PM",US Sassuolo,Atalanta BC,0,2,Alessio Dionisi,Gian Piero Gasperini,4-2-3-1
6,4103477,"1. Matchday|Sun, 20/08/23| 8:45 PM",US Lecce,SS Lazio,2,1,Roberto D'Aversa,Maurizio Sarri,4-3-3 Attacking
7,4103481,"1. Matchday|Sun, 20/08/23| 8:45 PM",Udinese Calcio,Juventus FC,0,3,Andrea Sottil,Massimiliano Allegri,3-5-2 flat
8,4103480,"1. Matchday|Mon, 21/08/23| 6:30 PM",Torino FC,Cagliari Calcio,0,0,Ivan Juric,Claudio Ranieri,3-4-2-1
9,4103472,"1. Matchday|Mon, 21/08/23| 8:45 PM",Bologna FC 1909,AC Milan,0,2,Thiago Motta,Stefano Pioli,4-2-3-1


In [46]:
# ── C.5  POC quality check ────────────────────────────────────────────────────
# Check how many rows have None managers — should be near zero for a recent season.
# High None% suggests selector needs fixing before running at scale.

stats = pd.read_sql(
    """
    SELECT
        COUNT(*)                                            AS total_matches,
        SUM(CASE WHEN home_manager IS NULL THEN 1 ELSE 0 END) AS missing_home_mgr,
        SUM(CASE WHEN away_manager IS NULL THEN 1 ELSE 0 END) AS missing_away_mgr,
        SUM(CASE WHEN attendance  IS NULL THEN 1 ELSE 0 END) AS missing_attendance
    FROM matches
    WHERE league_code='IT1' AND season=2023
    """,
    con
)
print(stats.to_string(index=False))
print()
print("Manager fill rate:",
      f"{(1 - stats['missing_home_mgr'][0] / stats['total_matches'][0]):.1%}")

 total_matches  missing_home_mgr  missing_away_mgr  missing_attendance
           380                 0                 1                   0

Manager fill rate: 100.0%


## Block D — Stage 2: last 15 seasons, all leagues

First substantial dataset — covers 2010–2025 for all 8 leagues. Run this after Block C confirms the parser is working correctly.
Expected runtime: 2–4 hours. Fully resumable if interrupted.

> **Tip:** run this overnight. The `log` output will show progress every 50 matches.

In [48]:
# ── D.1  Scrape 2025, all 8 leagues ──────────────────────────────────────────

summary_d1 = []
total_leagues = len(LEAGUES)

for i, league_code in enumerate(LEAGUES, 1):
    print(f"\n[{i}/{total_leagues}] Starting {league_code} 2025...")
    result = scrape_season(league_code, 2025, con)
    summary_d1.append(result)
    print(f"[{i}/{total_leagues}] {league_code} 2025 done — {result['success']} ok, {result['failed']} failed")

print("\n=== D.1 Complete ===")
pd.DataFrame(summary_d1)

12:32:26  INFO  GB1 2025: 380 matches to scrape



[1/8] Starting GB1 2025...


12:34:38  INFO    50/380 done  (50 ok, 0 failed)
12:37:17  INFO    100/380 done  (100 ok, 0 failed)
12:39:33  INFO    150/380 done  (150 ok, 0 failed)
12:41:48  INFO    200/380 done  (200 ok, 0 failed)
12:43:56  INFO    250/380 done  (250 ok, 0 failed)
12:46:03  INFO    300/380 done  (300 ok, 0 failed)
12:48:11  INFO    350/380 done  (350 ok, 0 failed)
12:49:29  INFO  GB1 2025: finished — 380 ok, 0 failed
12:49:29  INFO  IT1 2025: 380 matches to scrape


[1/8] GB1 2025 done — 380 ok, 0 failed

[2/8] Starting IT1 2025...


12:51:42  INFO    50/380 done  (50 ok, 0 failed)
12:53:56  INFO    100/380 done  (100 ok, 0 failed)
12:56:35  INFO    150/380 done  (150 ok, 0 failed)
12:58:55  INFO    200/380 done  (200 ok, 0 failed)
13:01:06  INFO    250/380 done  (250 ok, 0 failed)
13:03:46  INFO    300/380 done  (300 ok, 0 failed)
13:06:09  INFO    350/380 done  (350 ok, 0 failed)
13:07:31  INFO  IT1 2025: finished — 380 ok, 0 failed
13:07:31  INFO  ES1 2025: 380 matches to scrape


...

14:12:59  WARNING  Attempt 1/3 failed for https://www.transfermarkt.com/spielbericht/index/spielbericht/4641416: HTTPSConnectionPool(host='www.transfermarkt.com', port=443): Read timed out. (read timeout=15)
14:13:17  WARNING  Attempt 2/3 failed for https://www.transfermarkt.com/spielbericht/index/spielbericht/4641416: HTTPSConnectionPool(host='www.transfermarkt.com', port=443): Read timed out. (read timeout=15)
14:15:25  INFO    50/306 done  (50 ok, 0 failed)
14:17:56  INFO    100/306 done  (100 ok, 0 failed)
14:20:49  INFO    150/306 done  (150 ok, 0 failed)
14:23:13  INFO    200/306 done  (200 ok, 0 failed)
14:25:35  INFO    250/306 done  (250 ok, 0 failed)
14:27:45  INFO    300/306 done  (300 ok, 0 failed)
14:28:03  INFO  NL1 2025: finished — 306 ok, 0 failed
14:28:03  INFO  TR1 2025: 306 matches to scrape


[7/8] NL1 2025 done — 306 ok, 0 failed

[8/8] Starting TR1 2025...


14:30:18  INFO    50/306 done  (50 ok, 0 failed)
14:32:30  INFO    100/306 done  (100 ok, 0 failed)
14:35:07  INFO    150/306 done  (150 ok, 0 failed)
14:37:24  INFO    200/306 done  (200 ok, 0 failed)
14:39:37  INFO    250/306 done  (250 ok, 0 failed)
14:42:07  INFO    300/306 done  (300 ok, 0 failed)
14:42:22  INFO  TR1 2025: finished — 306 ok, 0 failed


[8/8] TR1 2025 done — 306 ok, 0 failed

=== D.1 Complete ===


,league,season,total,success,failed
0,GB1,2025,380,380,0
1,IT1,2025,380,380,0
2,ES1,2025,380,380,0
3,L1,2025,306,306,0
4,FR1,2025,306,306,0
5,PO1,2025,306,306,0
6,NL1,2025,306,306,0
7,TR1,2025,306,306,0


In [50]:
# ── D.2  Scrape 2024, all 8 leagues ──────────────────────────────────────────

summary_d2 = []
total_leagues = len(LEAGUES)

for i, league_code in enumerate(LEAGUES, 1):
    print(f"\n[{i}/{total_leagues}] Starting {league_code} 2024...")
    result = scrape_season(league_code, 2024, con)
    summary_d2.append(result)
    print(f"[{i}/{total_leagues}] {league_code} 2024 done — {result['success']} ok, {result['failed']} failed")

print("\n=== D.2 Complete ===")
pd.DataFrame(summary_d2)

14:53:52  INFO  GB1 2024: 380 matches to scrape



[1/8] Starting GB1 2024...


14:55:41  INFO    50/380 done  (50 ok, 0 failed)
14:57:33  INFO    100/380 done  (100 ok, 0 failed)
14:59:45  INFO    150/380 done  (150 ok, 0 failed)
15:01:25  INFO    200/380 done  (200 ok, 0 failed)
15:03:35  INFO    250/380 done  (250 ok, 0 failed)
15:05:41  INFO    300/380 done  (300 ok, 0 failed)
15:07:44  INFO    350/380 done  (350 ok, 0 failed)
15:08:46  INFO  GB1 2024: finished — 380 ok, 0 failed
15:08:46  INFO  IT1 2024: 380 matches to scrape


[1/8] GB1 2024 done — 380 ok, 0 failed

[2/8] Starting IT1 2024...


15:10:26  INFO    50/380 done  (50 ok, 0 failed)
15:12:16  INFO    100/380 done  (100 ok, 0 failed)
15:14:18  INFO    150/380 done  (150 ok, 0 failed)
15:16:04  INFO    200/380 done  (200 ok, 0 failed)
15:18:12  INFO    250/380 done  (250 ok, 0 failed)
15:19:59  INFO    300/380 done  (300 ok, 0 failed)
15:21:58  INFO    350/380 done  (350 ok, 0 failed)
15:23:16  INFO  IT1 2024: finished — 380 ok, 0 failed
15:23:16  INFO  ES1 2024: 380 matches to scrape


...

16:15:29  INFO    50/306 done  (50 ok, 0 failed)
16:17:14  INFO    100/306 done  (100 ok, 0 failed)
16:18:51  INFO    150/306 done  (150 ok, 0 failed)
16:20:24  INFO    200/306 done  (200 ok, 0 failed)
16:22:27  INFO    250/306 done  (250 ok, 0 failed)
16:24:19  INFO    300/306 done  (300 ok, 0 failed)
16:24:30  INFO  NL1 2024: finished — 306 ok, 0 failed
16:24:30  INFO  TR1 2024: 342 matches to scrape


[7/8] NL1 2024 done — 306 ok, 0 failed

[8/8] Starting TR1 2024...


16:26:20  INFO    50/342 done  (50 ok, 0 failed)
16:28:09  INFO    100/342 done  (100 ok, 0 failed)
16:30:08  INFO    150/342 done  (150 ok, 0 failed)
16:31:49  INFO    200/342 done  (200 ok, 0 failed)
16:33:34  INFO    250/342 done  (250 ok, 0 failed)
16:35:25  INFO    300/342 done  (300 ok, 0 failed)
16:37:06  INFO  TR1 2024: finished — 342 ok, 0 failed


[8/8] TR1 2024 done — 342 ok, 0 failed

=== D.2 Complete ===


,league,season,total,success,failed
0,GB1,2024,380,380,0
1,IT1,2024,380,380,0
2,ES1,2024,380,380,0
3,L1,2024,306,306,0
4,FR1,2024,306,306,0
5,PO1,2024,306,306,0
6,NL1,2024,306,306,0
7,TR1,2024,342,342,0


In [51]:
print(f"SLEEP_BASE   = {SLEEP_BASE}")
print(f"SLEEP_JITTER = {SLEEP_JITTER}")
print()

pd.read_sql("""
    SELECT league_code,
           COUNT(DISTINCT season) as seasons,
           COUNT(*) as matches
    FROM matches
    GROUP BY league_code
    ORDER BY league_code
""", con)

SLEEP_BASE   = 1.0
SLEEP_JITTER = 0.2



,league_code,seasons,matches
0,ES1,2,760
1,FR1,2,612
2,GB1,2,760
3,IT1,3,1140
4,L1,2,612
5,NL1,2,612
6,PO1,2,612
7,TR1,2,648


In [52]:
# ── D.3  Scrape 2023, all 8 leagues ──────────────────────────────────────────

summary_d3 = []
total_leagues = len(LEAGUES)

for i, league_code in enumerate(LEAGUES, 1):
    print(f"\n[{i}/{total_leagues}] Starting {league_code} 2023...")
    result = scrape_season(league_code, 2023, con)
    summary_d3.append(result)
    print(f"[{i}/{total_leagues}] {league_code} 2023 done — {result['success']} ok, {result['failed']} failed")

print("\n=== D.3 Complete ===")
pd.DataFrame(summary_d3)

16:38:48  INFO  GB1 2023: 380 matches to scrape



[1/8] Starting GB1 2023...


16:40:32  INFO    50/380 done  (50 ok, 0 failed)
16:42:19  INFO    100/380 done  (100 ok, 0 failed)
16:44:19  INFO    150/380 done  (150 ok, 0 failed)
16:46:07  INFO    200/380 done  (200 ok, 0 failed)
16:48:25  INFO    250/380 done  (250 ok, 0 failed)
16:50:00  INFO    300/380 done  (300 ok, 0 failed)
16:51:41  INFO    350/380 done  (350 ok, 0 failed)
16:52:46  INFO  GB1 2023: finished — 380 ok, 0 failed
16:52:46  INFO  IT1 2023: 0 matches to scrape
16:52:46  INFO  IT1 2023: finished — 0 ok, 0 failed
16:52:46  INFO  ES1 2023: 380 matches to scrape


[1/8] GB1 2023 done — 380 ok, 0 failed

[2/8] Starting IT1 2023...
[2/8] IT1 2023 done — 0 ok, 0 failed

[3/8] Starting ES1 2023...


16:54:27  INFO    50/380 done  (50 ok, 0 failed)
16:56:19  INFO    100/380 done  (100 ok, 0 failed)
16:57:59  INFO    150/380 done  (150 ok, 0 failed)
16:59:42  INFO    200/380 done  (200 ok, 0 failed)
17:01:25  INFO    250/380 done  (250 ok, 0 failed)
17:03:17  INFO    300/380 done  (300 ok, 0 failed)
17:05:17  INFO    350/380 done  (350 ok, 0 failed)
17:06:26  INFO  ES1 2023: finished — 380 ok, 0 failed
17:06:26  INFO  L1 2023: 306 matches to scrape


...

17:42:29  INFO    50/306 done  (50 ok, 0 failed)
17:44:12  INFO    100/306 done  (100 ok, 0 failed)
17:46:02  INFO    150/306 done  (150 ok, 0 failed)
17:47:51  INFO    200/306 done  (200 ok, 0 failed)
17:49:28  INFO    250/306 done  (250 ok, 0 failed)
17:51:08  INFO    300/306 done  (300 ok, 0 failed)
17:51:19  INFO  NL1 2023: finished — 306 ok, 0 failed
17:51:19  INFO  TR1 2023: 380 matches to scrape


[7/8] NL1 2023 done — 306 ok, 0 failed

[8/8] Starting TR1 2023...


17:52:54  INFO    50/380 done  (50 ok, 0 failed)
17:54:39  INFO    100/380 done  (100 ok, 0 failed)
17:56:23  INFO    150/380 done  (150 ok, 0 failed)
17:57:59  INFO    200/380 done  (200 ok, 0 failed)
17:59:35  INFO    250/380 done  (250 ok, 0 failed)
18:01:19  INFO    300/380 done  (300 ok, 0 failed)
18:02:59  INFO    350/380 done  (350 ok, 0 failed)
18:04:05  INFO  TR1 2023: finished — 380 ok, 0 failed


[8/8] TR1 2023 done — 380 ok, 0 failed

=== D.3 Complete ===


,league,season,total,success,failed
0,GB1,2023,380,380,0
1,IT1,2023,0,0,0
2,ES1,2023,380,380,0
3,L1,2023,306,306,0
4,FR1,2023,306,306,0
5,PO1,2023,306,306,0
6,NL1,2023,306,306,0
7,TR1,2023,380,380,0


In [53]:
# ── OPTION A: Load transfermarkt-datasets into DB (2012-2022) ─────────────────
# Run this cell once. Inserts ~30k rows in ~5 minutes, no scraping needed.
# Uses INSERT OR IGNORE so already-scraped matches are never overwritten.

import requests, io, pandas as pd, sqlite3
from datetime import datetime

TM_DATASET_URL = "https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/games.csv.gz"

OPTION_A_LEAGUES  = set(LEAGUES.keys())          # all 8 leagues in scope
OPTION_A_SEASONS  = set(range(2012, 2023))        # 2012-2022 (2023-2025 already scraped)

print("Downloading transfermarkt-datasets games.csv...")
response = requests.get(TM_DATASET_URL, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()
tm = pd.read_csv(io.BytesIO(response.content), compression="gzip")
print(f"Downloaded: {len(tm):,} rows")

# ── Filter to our leagues and seasons ────────────────────────────────────────
tm_filtered = tm[
    tm["competition_id"].isin(OPTION_A_LEAGUES) &
    tm["season"].isin(OPTION_A_SEASONS)
].copy()
print(f"After filter: {len(tm_filtered):,} rows")

# ── Map columns to our schema ─────────────────────────────────────────────────
# tm-datasets uses competition_id for league_code
# game_id is numeric in both datasets — should join cleanly
now = datetime.utcnow().isoformat()

rows = []
for _, r in tm_filtered.iterrows():
    rows.append({
        "game_id"        : str(int(r["game_id"])),
        "league_code"    : r["competition_id"],
        "season"         : int(r["season"]),
        "round"          : r.get("round", None),
        "date"           : r.get("date", None),
        "home_club"      : r.get("home_club_name", None),
        "away_club"      : r.get("away_club_name", None),
        "home_goals"     : int(r["home_club_goals"]) if pd.notna(r.get("home_club_goals")) else None,
        "away_goals"     : int(r["away_club_goals"]) if pd.notna(r.get("away_club_goals")) else None,
        "home_manager"   : r.get("home_club_manager_name", None),
        "away_manager"   : r.get("away_club_manager_name", None),
        "home_formation" : r.get("home_club_formation", None),
        "away_formation" : r.get("away_club_formation", None),
        "attendance"     : int(r["attendance"]) if pd.notna(r.get("attendance")) else None,
        "referee"        : r.get("referee", None),
        "scraped_at"     : now,
    })

# ── Insert into DB ────────────────────────────────────────────────────────────
INSERT_SQL = """
INSERT OR IGNORE INTO matches
    (game_id, league_code, season, round, date,
     home_club, away_club, home_goals, away_goals,
     home_manager, away_manager,
     home_formation, away_formation,
     attendance, referee, scraped_at)
VALUES
    (:game_id, :league_code, :season, :round, :date,
     :home_club, :away_club, :home_goals, :away_goals,
     :home_manager, :away_manager,
     :home_formation, :away_formation,
     :attendance, :referee, :scraped_at)
"""

cur = con.cursor()
cur.executemany(INSERT_SQL, rows)
con.commit()
print(f"Inserted: {cur.rowcount:,} new rows (skipped duplicates)")

# ── Verify ────────────────────────────────────────────────────────────────────
print()
pd.read_sql("""
    SELECT league_code,
           MIN(season) AS first_season,
           MAX(season) AS last_season,
           COUNT(*)    AS total_matches,
           ROUND(AVG(CASE WHEN home_manager IS NOT NULL THEN 1.0 ELSE 0 END) * 100, 1)
               AS manager_fill_pct
    FROM matches
    GROUP BY league_code
    ORDER BY league_code
""", con)




Downloaded: 88,783 rows
After filter: 30,101 rows
Inserted: 30,101 new rows (skipped duplicates)



,league_code,first_season,last_season,total_matches,manager_fill_pct
0,ES1,2012,2025,5320,100.0
1,FR1,2012,2025,4997,100.0
2,GB1,2012,2025,5320,100.0
3,IT1,2012,2025,5320,100.0
4,L1,2012,2025,4284,100.0
5,NL1,2012,2025,4210,100.0
6,PO1,2012,2025,4152,99.9
7,TR1,2012,2025,4618,99.3


In [54]:
# How many game_index entries are still pending (not in matches table)?
pd.read_sql("""
    SELECT gi.league_code,
           COUNT(*) as pending
    FROM game_index gi
    LEFT JOIN matches m USING (game_id)
    WHERE m.game_id IS NULL
    GROUP BY gi.league_code
    ORDER BY gi.league_code
""", con)

,league_code,pending
0,ES1,7384
1,FR1,6850
2,GB1,7384
3,IT1,6406
4,L1,5814
5,NL1,5814
6,PO1,5418
7,TR1,5748


In [ ]:
# ── OPTION B — Cell 2a: Setup + validation ───────────────────────────────────

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

#SCRAPERAPI_KEY     = "MyK"   # ← paste your key here
SCRAPERAPI_URL     = "http://api.scraperapi.com"
SCRAPERAPI_WORKERS = 5                     # concurrent requests

# ── ScraperAPI fetch function ─────────────────────────────────────────────────

def fetch_via_scraperapi(url: str, retries: int = 3) -> BeautifulSoup | None:
    """Route a request through ScraperAPI for proxy-rotated fetching."""
    params = {
        "api_key": SCRAPERAPI_KEY,
        "url"    : url,
        "render" : "false",
    }
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(SCRAPERAPI_URL, params=params, timeout=30)
            resp.raise_for_status()
            return BeautifulSoup(resp.content, "html.parser")
        except requests.RequestException as e:
            log.warning(f"ScraperAPI attempt {attempt}/{retries} failed for {url}: {e}")
            if attempt < retries:
                time.sleep(2 * attempt)
    log.error(f"ScraperAPI: all {retries} attempts failed for {url}")
    return None


def scrape_season_parallel(league_code: str, season: int,
                            con: sqlite3.Connection,
                            workers: int = SCRAPERAPI_WORKERS) -> dict:
    """Scrape all pending matches using a thread pool.
    Each worker opens its own DB connection to avoid SQLite threading issues.
    """
    pending = pd.read_sql("""
        SELECT gi.game_id, gi.league_code, gi.season
        FROM game_index gi
        LEFT JOIN matches m USING (game_id)
        WHERE gi.league_code = ? AND gi.season = ? AND m.game_id IS NULL
    """, con, params=(league_code, season))

    total   = len(pending)
    success = 0
    failed  = 0
    lock    = threading.Lock()

    log.info(f"{league_code} {season}: {total} matches to scrape ({workers} workers)")

    def scrape_one(row):
        # Each thread gets its own connection
        thread_con = sqlite3.connect(DB_PATH)
        try:
            url  = f"{BASE_URL}/spielbericht/index/spielbericht/{row.game_id}"
            soup = fetch_via_scraperapi(url)
            if soup is None:
                return False
            match_row = parse_match_report(soup, row.game_id,
                                           row.league_code, int(row.season))
            if match_row is None:
                return False
            with lock:
                thread_con.execute(INSERT_SQL, match_row)
                thread_con.commit()
            return True
        except Exception as e:
            log.warning(f"scrape_one failed for {row.game_id}: {e}")
            return False
        finally:
            thread_con.close()

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(scrape_one, row): row
                   for row in pending.itertuples()}
        done = 0
        for future in as_completed(futures):
            done += 1
            if future.result():
                success += 1
            else:
                failed += 1
            if done % 50 == 0:
                log.info(f"  {done}/{total} done  ({success} ok, {failed} failed)")

    log.info(f"{league_code} {season}: finished — {success} ok, {failed} failed")
    return {"league": league_code, "season": season,
            "total": total, "success": success, "failed": failed}


    def scrape_one(row):
        url  = f"{BASE_URL}/spielbericht/index/spielbericht/{row.game_id}"
        soup = fetch_via_scraperapi(url)
        if soup is None:
            return False
        match_row = parse_match_report(soup, row.game_id, row.league_code, int(row.season))
        if match_row is None:
            return False
        with lock:
            con.execute(INSERT_SQL, match_row)
            con.commit()
        return True

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(scrape_one, row): row
                   for row in pending.itertuples()}
        done = 0
        for future in as_completed(futures):
            done += 1
            if future.result():
                success += 1
            else:
                failed += 1
            if done % 50 == 0:
                log.info(f"  {done}/{total} done  ({success} ok, {failed} failed)")

    log.info(f"{league_code} {season}: finished — {success} ok, {failed} failed")
    return {"league": league_code, "season": season,
            "total": total, "success": success, "failed": failed}


# ── Validation: 10 matches from IT1 2011 ─────────────────────────────────────

print("Testing ScraperAPI on 10 matches from IT1 2011...")

test_matches = pd.read_sql("""
    SELECT gi.game_id, gi.league_code, gi.season
    FROM game_index gi
    LEFT JOIN matches m USING (game_id)
    WHERE gi.league_code = 'IT1' AND gi.season = 2011 AND m.game_id IS NULL
    LIMIT 10
""", con)

test_results = []
for row in test_matches.itertuples():
    url  = f"{BASE_URL}/spielbericht/index/spielbericht/{row.game_id}"
    soup = fetch_via_scraperapi(url)
    if soup:
        match_row = parse_match_report(soup, row.game_id, row.league_code, int(row.season))
        con.execute(INSERT_SQL, match_row)
        con.commit()
        test_results.append({"game_id": row.game_id, "ok": True,
                              "home_manager": match_row["home_manager"],
                              "away_manager": match_row["away_manager"]})
    else:
        test_results.append({"game_id": row.game_id, "ok": False,
                              "home_manager": None, "away_manager": None})

print(pd.DataFrame(test_results))

Testing ScraperAPI on 10 matches from IT1 2011...
   game_id    ok          home_manager          away_manager
0  1133602  True  Massimiliano Allegri          Edoardo Reja
1  1133596  True       Marco Giampaolo       Walter Mazzarri
2  1133600  True         Antonio Conte        Franco Colomba
3  1133595  True     Vincenzo Montella      Giuseppe Sannino
4  1133597  True     Domenico Di Carlo        Attilio Tesser
5  1133598  True     Sinisa Mihajlovic      Pierpaolo Bisoli
6  1133599  True      Alberto Malesani    Stefano Colantuono
7  1133601  True  Eusebio Di Francesco    Francesco Guidolin
8  1133604  True          Luis Enrique    Massimo Ficcadenti
9  1133603  True          Devis Mangia  Gian Piero Gasperini


In [ ]:
# ── OPTION B — Cell 2b: Full pre-2012 scrape ─────────────────────────────────

PRE_2012_SEASONS = list(range(2011, 1992, -1))   # newest first

summary_pre2012 = []
total_leagues = len(LEAGUES)

for i, league_code in enumerate(LEAGUES, 1):
    for season in PRE_2012_SEASONS:
        print(f"\n[{i}/{total_leagues}] {league_code} {season}...", flush=True)
        t_start = time.time()
        result  = scrape_season_parallel(league_code, season, con,
                                         workers=SCRAPERAPI_WORKERS)
        elapsed = time.time() - t_start
        rate    = result['total'] / elapsed if elapsed > 0 else 0
        summary_pre2012.append(result)
        print(
            f"  ✓ {result['success']} ok, {result['failed']} failed | "
            f"{elapsed/60:.1f} min | {rate:.1f} matches/sec",
            flush=True
        )

print("\n=== Pre-2012 scrape complete ===")
pd.DataFrame(summary_pre2012)


[1/8] GB1 2011...


18:48:27  INFO  GB1 2011: 380 matches to scrape (5 workers)
18:48:58  INFO    50/380 done  (50 ok, 0 failed)
18:49:28  INFO    100/380 done  (100 ok, 0 failed)
18:49:59  INFO    150/380 done  (150 ok, 0 failed)
18:50:29  INFO    200/380 done  (200 ok, 0 failed)


---

## Kernel restarted

In [ ]:
# ── Cell 2b — Pre-2012 scrape, 4 seasons (~2 hours) ──────────────────────────

import sqlite3
import time
import threading
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd

# ── Config (must match your session) ─────────────────────────────────────────
DB_PATH            = Path("tm_football.db")
BASE_URL           = "https://www.transfermarkt.com"
SCRAPERAPI_KEY     = "MyK"
SCRAPERAPI_URL     = "http://api.scraperapi.com"
SCRAPERAPI_WORKERS = 5

LEAGUES = {
    "GB1": "premier-league",
    "IT1": "serie-a",
    "ES1": "laliga",
    "L1" : "bundesliga",
    "FR1": "ligue-1",
    "PO1": "liga-nos",
    "NL1": "eredivisie",
    "TR1": "super-lig",
}

log = logging.getLogger(__name__)

# ── INSERT SQL ────────────────────────────────────────────────────────────────
INSERT_SQL = """
INSERT OR IGNORE INTO matches
    (game_id, league_code, season, round, date,
     home_club, away_club, home_goals, away_goals,
     home_manager, away_manager,
     home_formation, away_formation,
     attendance, referee, scraped_at)
VALUES
    (:game_id, :league_code, :season, :round, :date,
     :home_club, :away_club, :home_goals, :away_goals,
     :home_manager, :away_manager,
     :home_formation, :away_formation,
     :attendance, :referee, :scraped_at)
"""

# ── Parser ────────────────────────────────────────────────────────────────────
from datetime import datetime

def parse_match_report(soup, game_id, league_code, season):
    home_manager = away_manager = None
    manager_rows = []
    for td in soup.find_all("div", string=lambda t: t and "Manager:" in t):
        tr = td.parent.parent
        name_td = tr.find("td", class_="bench-table__td")
        if name_td:
            a = name_td.find("a")
            if a:
                manager_rows.append(a.get_text(strip=True))
    if len(manager_rows) > 0:
        home_manager = manager_rows[0]
    if len(manager_rows) > 1:
        away_manager = manager_rows[1]

    home_formation = away_formation = None
    formation_divs = soup.select("div.formation-subtitle")
    def extract_formation(text):
        return text.split(":")[-1].strip() if ":" in text else text.strip()
    if len(formation_divs) > 0:
        home_formation = extract_formation(formation_divs[0].get_text(strip=True))
    if len(formation_divs) > 1:
        away_formation = extract_formation(formation_divs[1].get_text(strip=True))

    attendance = None
    for strong in soup.find_all("strong"):
        text = strong.get_text(strip=True)
        if "Attendance" in text or "Zuschauer" in text:
            raw = text.split(":")[-1].strip()
            try:
                attendance = int(raw.replace(".", "").replace(",", ""))
            except ValueError:
                pass
            break

    referee = None
    for strong in soup.find_all("strong"):
        if "Referee" in strong.get_text() or "Schiedsrichter" in strong.get_text():
            for sib in strong.next_siblings:
                if hasattr(sib, "select_one"):
                    a = sib.select_one("a")
                    if a:
                        referee = a.get_text(strip=True)
                        break
            break

    home_goals = away_goals = None
    score_tag = soup.select_one("div.sb-endstand")
    if score_tag:
        score_text = score_tag.get_text(strip=True).split("(")[0].strip()
        if ":" in score_text:
            parts = score_text.split(":")
            try:
                home_goals = int(parts[0])
                away_goals = int(parts[1])
            except ValueError:
                pass

    home_club = away_club = None
    verein_links = soup.find_all(
        "a", href=lambda h: h and "/startseite/verein/" in h
    )
    name_links = [a for a in verein_links if a.get_text(strip=True)]
    if len(name_links) > 0:
        home_club = name_links[0].get_text(strip=True)
    if len(name_links) > 1:
        away_club = name_links[1].get_text(strip=True)

    date = None
    date_tag = soup.select_one("p.sb-datum")
    if date_tag:
        date = date_tag.get_text(strip=True)

    return {
        "game_id"        : game_id,
        "league_code"    : league_code,
        "season"         : season,
        "round"          : None,
        "date"           : date,
        "home_club"      : home_club,
        "away_club"      : away_club,
        "home_goals"     : home_goals,
        "away_goals"     : away_goals,
        "home_manager"   : home_manager,
        "away_manager"   : away_manager,
        "home_formation" : home_formation,
        "away_formation" : away_formation,
        "attendance"     : attendance,
        "referee"        : referee,
        "scraped_at"     : datetime.utcnow().isoformat(),
    }

# ── ScraperAPI fetch ──────────────────────────────────────────────────────────
def fetch_via_scraperapi(url, retries=3):
    params = {
        "api_key": SCRAPERAPI_KEY,
        "url"    : url,
        "render" : "false",
    }
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(SCRAPERAPI_URL, params=params, timeout=30)
            resp.raise_for_status()
            return BeautifulSoup(resp.content, "html.parser")
        except requests.RequestException as e:
            log.warning(f"ScraperAPI attempt {attempt}/{retries} failed for {url}: {e}")
            if attempt < retries:
                time.sleep(2 * attempt)
    log.error(f"ScraperAPI: all {retries} attempts failed for {url}")
    return None

# ── Parallel scrape ───────────────────────────────────────────────────────────
def scrape_season_parallel(league_code, season, workers=SCRAPERAPI_WORKERS):
    con = sqlite3.connect(DB_PATH)
    pending = pd.read_sql("""
        SELECT gi.game_id, gi.league_code, gi.season
        FROM game_index gi
        LEFT JOIN matches m USING (game_id)
        WHERE gi.league_code = ? AND gi.season = ? AND m.game_id IS NULL
    """, con, params=(league_code, season))
    con.close()

    total   = len(pending)
    success = 0
    failed  = 0
    lock    = threading.Lock()

    log.info(f"{league_code} {season}: {total} matches to scrape ({workers} workers)")

    def scrape_one(row):
        url  = f"{BASE_URL}/spielbericht/index/spielbericht/{row.game_id}"
        soup = fetch_via_scraperapi(url)
        if soup is None:
            return False
        match_row = parse_match_report(soup, row.game_id,
                                       row.league_code, int(row.season))
        if match_row is None:
            return False
        thread_con = sqlite3.connect(DB_PATH)
        try:
            with lock:
                thread_con.execute(INSERT_SQL, match_row)
                thread_con.commit()
            return True
        except Exception as e:
            log.warning(f"DB write failed for {row.game_id}: {e}")
            return False
        finally:
            thread_con.close()

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(scrape_one, row): row
                   for row in pending.itertuples()}
        done = 0
        for future in as_completed(futures):
            done += 1
            if future.result():
                success += 1
            else:
                failed += 1
            if done % 50 == 0:
                log.info(f"  {done}/{total} done  ({success} ok, {failed} failed)")

    log.info(f"{league_code} {season}: finished — {success} ok, {failed} failed")
    return {"league": league_code, "season": season,
            "total": total, "success": success, "failed": failed}

# ── Run ───────────────────────────────────────────────────────────────────────
PRE_2012_SEASONS  = [2011, 2010, 2009, 2008]
summary_pre2012   = []
total_leagues     = len(LEAGUES)

for i, league_code in enumerate(LEAGUES, 1):
    for season in PRE_2012_SEASONS:
        print(f"\n[{i}/{total_leagues}] {league_code} {season}...", flush=True)
        t_start = time.time()
        result  = scrape_season_parallel(league_code, season)
        elapsed = time.time() - t_start
        rate    = result['total'] / elapsed if elapsed > 0 else 0
        summary_pre2012.append(result)
        print(
            f"  ✓ {result['success']} ok, {result['failed']} failed | "
            f"{elapsed/60:.1f} min | {rate:.1f} matches/sec",
            flush=True
        )

print("\n=== Done ===")
pd.DataFrame(summary_pre2012)

  ✓ 380 ok, 0 failed | 4.6 min | 1.4 matches/sec

[2/8] IT1 2011...
  ✓ 360 ok, 0 failed | 4.1 min | 1.5 matches/sec

[2/8] IT1 2010...
  ✓ 380 ok, 0 failed | 4.1 min | 1.5 matches/sec

[2/8] IT1 2009...


ScraperAPI attempt 1/3 failed for https://www.transfermarkt.com/spielbericht/index/spielbericht/964557: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


  ✓ 380 ok, 0 failed | 4.2 min | 1.5 matches/sec

[2/8] IT1 2008...
  ✓ 380 ok, 0 failed | 4.2 min | 1.5 matches/sec

[3/8] ES1 2011...
  ✓ 380 ok, 0 failed | 4.1 min | 1.5 matches/sec

[3/8] ES1 2010...
  ✓ 380 ok, 0 failed | 4.2 min | 1.5 matches/sec

[3/8] ES1 2009...
  ✓ 380 ok, 0 failed | 4.0 min | 1.6 matches/sec

[3/8] ES1 2008...
  ✓ 380 ok, 0 failed | 4.2 min | 1.5 matches/sec

[4/8] L1 2011...


ScraperAPI attempt 1/3 failed for https://www.transfermarkt.com/spielbericht/index/spielbericht/1121134: 403 Client Error: Forbidden for url: http://api.scraperapi.com/?api_key=a4a33b8edc51a2ee3e939bbc7b7e5c56&url=https%3A%2F%2Fwww.transfermarkt.com%2Fspielbericht%2Findex%2Fspielbericht%2F1121134&render=false
ScraperAPI attempt 1/3 failed for https://www.transfermarkt.com/spielbericht/index/spielbericht/1121138: 403 Client Error: Forbidden for url: http://api.scraperapi.com/?api_key=a4a33b8edc51a2ee3e939bbc7b7e5c56&url=https%3A%2F%2Fwww.transfermarkt.com%2Fspielbericht%2Findex%2Fspielbericht%2F1121138&render=false
ScraperAPI attempt 1/3 failed for https://www.transfermarkt.com/spielbericht/index/spielbericht/1121139: 403 Client Error: Forbidden for url: http://api.scraperapi.com/?api_key=a4a33b8edc51a2ee3e939bbc7b7e5c56&url=https%3A%2F%2Fwww.transfermarkt.com%2Fspielbericht%2Findex%2Fspielbericht%2F1121139&render=false
ScraperAPI attempt 2/3 failed for https://www.transfermarkt.com/spi

  ✓ 117 ok, 189 failed | 5.6 min | 0.9 matches/sec

[4/8] L1 2010...


ScraperAPI attempt 1/3 failed for https://www.transfermarkt.com/spielbericht/index/spielbericht/1029852: 403 Client Error: Forbidden for url: http://api.scraperapi.com/?api_key=a4a33b8edc51a2ee3e939bbc7b7e5c56&url=https%3A%2F%2Fwww.transfermarkt.com%2Fspielbericht%2Findex%2Fspielbericht%2F1029852&render=false
ScraperAPI attempt 1/3 failed for https://www.transfermarkt.com/spielbericht/index/spielbericht/1029855: 403 Client Error: Forbidden for url: http://api.scraperapi.com/?api_key=a4a33b8edc51a2ee3e939bbc7b7e5c56&url=https%3A%2F%2Fwww.transfermarkt.com%2Fspielbericht%2Findex%2Fspielbericht%2F1029855&render=false
ScraperAPI attempt 1/3 failed for https://www.transfermarkt.com/spielbericht/index/spielbericht/1029853: 403 Client Error: Forbidden for url: http://api.scraperapi.com/?api_key=a4a33b8edc51a2ee3e939bbc7b7e5c56&url=https%3A%2F%2Fwww.transfermarkt.com%2Fspielbericht%2Findex%2Fspielbericht%2F1029853&render=false
ScraperAPI attempt 1/3 failed for https://www.transfermarkt.com/spi

In [2]:
# ── DB Status Check ───────────────────────────────────────────────────────────

import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("tm_football.db")
con = sqlite3.connect(DB_PATH)

pd.read_sql("""
    SELECT league_code,
           MIN(season) AS first_season,
           MAX(season) AS last_season,
           COUNT(*)    AS total_matches,
           ROUND(AVG(CASE WHEN home_manager IS NOT NULL 
                     THEN 1.0 ELSE 0 END) * 100, 1) AS manager_fill_pct
    FROM matches
    GROUP BY league_code
    ORDER BY league_code
""", con)

,league_code,first_season,last_season,total_matches,manager_fill_pct
0,ES1,2008,2025,6840,92.1
1,FR1,2012,2025,4997,100.0
2,GB1,2008,2025,6840,99.6
3,IT1,2008,2025,6840,99.3
4,L1,2011,2025,4401,100.0
5,NL1,2012,2025,4210,100.0
6,PO1,2012,2025,4152,99.9
7,TR1,2012,2025,4618,99.3


## Block E — Stage 2: full archive (pre-2010)

Completes the data lake back to 1993. Run after Block D.
Expected runtime: 4–8 hours. Run overnight.

Going backwards from 2009 → 1993 means any interruption still leaves
you with a clean recent dataset from Block D intact.

In [ ]:
# ── E.1  Scrape 1993–2009, all leagues (reverse chronological) ────────────────

BLOCK_E_START = 1993
BLOCK_E_END   = 2009

summary_e = []
for league_code in LEAGUES:
    for season in range(BLOCK_E_END, BLOCK_E_START - 1, -1):   # newest first
        result = scrape_season(league_code, season, con)
        summary_e.append(result)

pd.DataFrame(summary_e)

## Block F — Database summary

Run at any point to check the current state of the data lake.

In [ ]:
# ── F.1  Coverage overview ────────────────────────────────────────────────────

print("=== game_index coverage ===")
display(pd.read_sql(
    """
    SELECT league_code,
           MIN(season) AS first_season,
           MAX(season) AS last_season,
           COUNT(*)    AS total_games
    FROM game_index
    GROUP BY league_code
    ORDER BY league_code
    """,
    con
))

print("\n=== matches scraped ===")
display(pd.read_sql(
    """
    SELECT league_code,
           MIN(season) AS first_season,
           MAX(season) AS last_season,
           COUNT(*)    AS scraped,
           ROUND(AVG(CASE WHEN home_manager IS NOT NULL THEN 1.0 ELSE 0 END) * 100, 1)
               AS manager_fill_pct
    FROM matches
    GROUP BY league_code
    ORDER BY league_code
    """,
    con
))

print("\n=== pending (indexed but not yet scraped) ===")
display(pd.read_sql(
    """
    SELECT gi.league_code, COUNT(*) AS pending
    FROM game_index gi
    LEFT JOIN matches m USING (game_id)
    WHERE m.game_id IS NULL
    GROUP BY gi.league_code
    ORDER BY gi.league_code
    """,
    con
))